**Generate Sample Data for a Simple Factor Model**

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500

# Randomize factor exposures. In a proper factor model, I will estimate a separate set of factor exposures (beta) for each company using historical returns and the Fama-French factors.
# Or using rolling regressions to generate a panel of factor exposures (beta)
df = pd.DataFrame({
    'MKT': np.random.normal(0, 1, n), # Market excess return
    'SMB': np.random.normal(0, 1, n), # Small minus big
    'HML': np.random.normal(0, 1, n), # Value
    'MOM': np.random.normal(0, 1, n) # Momentum
})

# Hypotherical ESG scores which partially correlate with the selected factors.
# Other risk metrics can be used. Using ESG scores is just for simplicity, dmonstration purpose.
df['ESG'] = (
    0.45 * df['SMB']
    - 0.35 * df['HML']
    + 0.22 * df['MOM']
    + np.random.normal(0, 0.5, n)
)

**Orthogonalize ESG by OLS Regression to Generate a Pure ESG Variable Uncorrelated to Value, Momentum and Size Factors**

In [ ]:
import statsmodels.api as sm

X = df[['MKT', 'SMB', 'HML', 'MOM']]
X = sm.add_constant(X)

y = df['ESG']

model = sm.OLS(y, X).fit() # Estimate the ESG explained by known factors. Without .fit(), you only define the regression model structure.

print(model.summary()) # Display regression results

df['ESG_orthogonal'] = model.resid # Store the regression residuals as the orthogonalized ESG factor, i.e. residuals = orthogonalized ESG factor.

                            OLS Regression Results                            
Dep. Variable:                    ESG   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.603
Method:                 Least Squares   F-statistic:                     190.7
Date:                Mon, 11 May 2026   Prob (F-statistic):           8.54e-99
Time:                        10:12:07   Log-Likelihood:                -339.11
No. Observations:                 500   AIC:                             688.2
Df Residuals:                     495   BIC:                             709.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0023      0.022     -0.106      0.9

**Validate Orthogonalized ESG with Factors**

In [ ]:
corrs = df[['ESG_orthogonal', 'MKT', 'SMB', 'HML', 'MOM']].corr()

print(corrs) # Check if there is significnat corelation

                ESG_orthogonal           MKT           SMB           HML  \
ESG_orthogonal    1.000000e+00  1.349275e-16 -5.763042e-16 -1.413804e-16   
MKT               1.349275e-16  1.000000e+00 -7.567075e-02 -5.779053e-02   
SMB              -5.763042e-16 -7.567075e-02  1.000000e+00  7.603839e-02   
HML              -1.413804e-16 -5.779053e-02  7.603839e-02  1.000000e+00   
MOM              -2.401855e-16  6.413987e-02 -2.157827e-02 -2.205658e-02   

                         MOM  
ESG_orthogonal -2.401855e-16  
MKT             6.413987e-02  
SMB            -2.157827e-02  
HML            -2.205658e-02  
MOM             1.000000e+00  


**Add Orthogonalized ESG to the Factor Model as a New Variable**

In [ ]:
# Simulated returns
df['Returns'] = (
    0.8 * df['MKT']
    + 0.2 * df['SMB']
    + 0.15 * df['ESG_orthogonal']
    + np.random.normal(0, 1, n)
)

X2 = df[['MKT', 'SMB', 'HML', 'MOM', 'ESG_orthogonal']]
X2 = sm.add_constant(X2)

y2 = df['Returns']

factor_model = sm.OLS(y2, X2).fit()

print(factor_model.summary())

                            OLS Regression Results                            
Dep. Variable:                Returns   R-squared:                       0.412
Model:                            OLS   Adj. R-squared:                  0.406
Method:                 Least Squares   F-statistic:                     69.30
Date:                Mon, 11 May 2026   Prob (F-statistic):           7.68e-55
Time:                        10:12:07   Log-Likelihood:                -711.46
No. Observations:                 500   AIC:                             1435.
Df Residuals:                     494   BIC:                             1460.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0194      0.045      0.